# Kalibrasi Judge FP_NLP — Runner Kaggle (issue #2)

Menjalankan judge produksi **Qwen2.5-7B-Instruct** pada 260 baris beranotasi manual, lalu
menghitung presisi/recall + interval Wilson.

Notebook ini **runner**, bukan artefak paper. Artefak paper-nya
`notebooks/revisi/08_kalibrasi_judge.ipynb` di repo, dijalankan ulang di laptop setelah hasil
notebook ini diunduh.

---

## Setelan Kaggle yang WAJIB dipasang dulu (panel kanan)

| Setelan | Nilai |
|---|---|
| **Accelerator** | `GPU T4 x2` |
| **Internet** | `On` (perlu untuk clone repo + unduh bobot model) |
| **Persistence** | boleh mati |

Kalau Accelerator masih `None`, sel 2 akan berhenti dan memberi tahu — bukan jalan diam-diam
di CPU lalu menggantung berjam-jam.

Setelah itu: **Run All**. Perkiraan 25-45 menit (unduh bobot ~15 GB memakan porsi terbesar).

## 1. Clone repo

In [4]:
import os, subprocess, sys, time
from pathlib import Path

REPO_URL = 'https://github.com/henray404/FP_NLP.git'
REPO = Path('/kaggle/working/FP_NLP')
BRANCH = 'main'   # ganti ke 'issue-2' kalau kerjanya belum di-merge

t0 = time.time()
if REPO.exists():
    print('repo sudah ada -> git pull')
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(REPO)],
                   check=True)

os.chdir(REPO); sys.path.insert(0, str(REPO))
print(f'\nrepo siap di {REPO} ({time.time()-t0:.0f}s)')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

# Data input WAJIB ada. Ia ter-track di git (bukan kena .gitignore), jadi ikut ter-clone.
INPUT = REPO / 'data/Final/easy_clean.jsonl'
assert INPUT.exists(), f'{INPUT} tidak ada — clone gagal atau branch salah'
print(f'{INPUT.name}: {sum(1 for _ in open(INPUT, encoding="utf-8")):,} baris')

repo sudah ada -> git pull
Fetching origin


Already on 'main'


Your branch is up to date with 'origin/main'.
Already up to date.

repo siap di /kaggle/working/FP_NLP (1s)
b15ca64 feat(kalibrasi): catat provenans run, uji chat template, silang bias format Q2

easy_clean.jsonl: 2,686 baris


## 2. Periksa GPU

Judge 7B fp16 butuh ~15 GB. Satu T4 (15 GB terpakai) tidak cukup — perlu **dua** T4, model
dibelah `device_map='auto'`. Sel ini berhenti kalau syaratnya tidak terpenuhi.

In [5]:
import torch

n_gpu = torch.cuda.device_count()
total_gb = sum(torch.cuda.get_device_properties(i).total_memory
               for i in range(n_gpu)) / 1e9 if n_gpu else 0

for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
print(f'total VRAM: {total_gb:.1f} GB')

if n_gpu == 0:
    raise SystemExit(
        'TIDAK ADA GPU. Panel kanan -> Accelerator -> "GPU T4 x2", lalu Run All lagi.')
if total_gb < 18:
    raise SystemExit(
        f'VRAM total cuma {total_gb:.1f} GB. Judge 7B fp16 butuh ~15 GB + ruang aktivasi. '
        'Pilih "GPU T4 x2" (bukan "GPU P100" atau "T4 x1").')
print('\nOK — cukup untuk judge 7B')

GPU 0: Tesla T4, 15.6 GB
GPU 1: Tesla T4, 15.6 GB
total VRAM: 31.3 GB

OK — cukup untuk judge 7B


## 3. Dependensi

Image Kaggle sudah punya torch, transformers, accelerate. Yang kurang cuma `langdetect`
(dipakai `src/preprop/filter_rules.py`). vLLM sengaja **tidak** dipasang: instalasinya sering
menarik ulang torch dan gagal di Kaggle, sementara untuk 520 generasi pendek keunggulan
kecepatannya tidak berarti. Backend `hf` memakai prompt, decoding greedy, dan parser vonis yang
sama persis dengan jalur vLLM produksi — modelnya pun sama.

In [6]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'langdetect'], check=True)

import importlib
for m in ['torch', 'transformers', 'accelerate', 'langdetect']:
    mod = importlib.import_module(m)
    print(f'{m:14} {getattr(mod, "__version__", "?")}')

torch          2.10.0+cu128
transformers   5.0.0
accelerate     1.13.0
langdetect     ?


## 4. Bangun ulang `easy_clean_v2.jsonl`

Deterministik. Angkanya wajib sama dengan acuan notebook 07 — kalau meleset, sampel 260 baris
tidak lagi menunjuk baris yang dianotasi dan seluruh kalibrasi jadi tak sahih. Assert di bawah
sengaja keras.

In [7]:
from src.preprop import clean_easy_v2
from src.data_validation import calibration_labels as labels

stats = clean_easy_v2.run()
for k, v in stats.items():
    print(f'{k:14}: {v}')

assert stats['input'] == clean_easy_v2.ACUAN_INPUT, stats['input']
assert stats['bersih'] == clean_easy_v2.ACUAN_KEEP_TOTAL, stats['bersih']
for alasan, n in clean_easy_v2.ACUAN_ALASAN.items():
    assert stats['alasan'][alasan] == n, (alasan, stats['alasan'][alasan], n)

sampel = labels.sampel_kalibrasi('data/Final/easy_clean_v2.jsonl')
assert len(sampel) == labels.UKURAN_SAMPEL
print(f'\ncocok dengan acuan notebook 07; sampel {len(sampel)} baris seed={labels.SEED} terbentuk')

input         : 2686
bersih        : 2336
dibuang       : 350
alasan        : {'KEEP': 2229, 'jawaban_bare_letter': 278, 'REPAIR': 107, 'cara_fabricated': 22, 'english_leak': 50}
output_path   : data/Final/easy_clean_v2.jsonl
dropped_path  : data/Final/easy_clean_v2_dropped.jsonl

cocok dengan acuan notebook 07; sampel 260 baris seed=7 terbentuk


## 5. Jalankan judge — tahap paling lama

520 vonis (260 baris x Q1+Q2). Unduh bobot ~15 GB sekali di awal.

Resumable: `.progress` ditulis + `fsync` tiap batch. Kalau sesi Kaggle mati di tengah,
jalankan ulang notebook — yang sudah dinilai tidak diulang.

In [8]:
from pathlib import Path
from src.data_validation import calibrate_judge as cal

OUT = Path('/kaggle/working/hasil_kalibrasi')
OUT.mkdir(parents=True, exist_ok=True)

t0 = time.time()
hasil = cal.run(
    Path('data/Final/easy_clean_v2.jsonl'),
    OUT,
    judge_backend='hf',
    judge_model='Qwen/Qwen2.5-7B-Instruct',   # judge produksi, sama dgn DEFAULT_VLLM_JUDGE
    device_map='auto',                        # belah ke 2xT4
    batch_size=16,
    regenerasi=False,                         # sudah dibangun di sel 4
)
print(f'\nselesai dalam {(time.time()-t0)/60:.1f} menit')

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[Q1] 260 baris, 0 sudah dinilai, 260 sisa
  [Q1] 16/260
  [Q1] 32/260
  [Q1] 48/260
  [Q1] 64/260
  [Q1] 80/260
  [Q1] 96/260
  [Q1] 112/260
  [Q1] 128/260
  [Q1] 144/260
  [Q1] 160/260
  [Q1] 176/260
  [Q1] 192/260
  [Q1] 208/260
  [Q1] 224/260
  [Q1] 240/260
  [Q1] 256/260
  [Q1] 260/260
[Q2] 260 baris, 0 sudah dinilai, 260 sisa
  [Q2] 16/260
  [Q2] 32/260
  [Q2] 48/260
  [Q2] 64/260
  [Q2] 80/260
  [Q2] 96/260
  [Q2] 112/260
  [Q2] 128/260
  [Q2] 144/260
  [Q2] 160/260
  [Q2] 176/260
  [Q2] 192/260
  [Q2] 208/260
  [Q2] 224/260
  [Q2] 240/260
  [Q2] 256/260
  [Q2] 260/260
KALIBRASI JUDGE vs ANOTASI MANUAL
model judge     : Qwen/Qwen2.5-7B-Instruct
sampel          : 260 baris (seed=7)
dinilai Q1/Q2   : 260 / 260
ditandai judge  : Q1 16, Q2 47

Tahap A (regex, sebelum judge): 9 baris terbuang, 0 di antaranya juga positif manual Q1

Q1 keterjawaban soal — 10 positif manual
  TP   7   FP   9   FN   3
  presisi 0.438  (n= 16)  CI95 Wilson [0.231, 0.668]   <- BATAS BAWAH (lihat catatan la

## 6. Angka untuk ditempel ke issue #2

Salin blok di bawah apa adanya ke komentar issue. Interval Wilson ikut — jangan kutip titik
estimasi sendirian.

In [9]:
q1, q2 = hasil['q1'], hasil['q2']
lo_p, hi_p = q1['presisi_ci']
lo_r, hi_r = q1['recall_ci']

print('```')
print('ANGKA UNTUK ISSUE #2')
print('model judge   : Qwen/Qwen2.5-7B-Instruct (backend hf, greedy, 2xT4)')
print(f"sampel        : {hasil['n_sampel']} baris, seed=7")
print(f"ditandai judge: Q1 {hasil['n_ditandai_judge_q1']}, Q2 {hasil['n_ditandai_judge_q2']}")
print()
print(f"Q1  TP {q1['tp']}  FP {q1['fp']}  FN {q1['fn']}")
print(f"Q1  presisi {q1['presisi']:.3f}  n={q1['presisi_n']}  CI95 [{lo_p:.3f}, {hi_p:.3f}]  (BATAS BAWAH)")
print(f"Q1  recall  {q1['recall']:.3f}  n={q1['recall_n']}  CI95 [{lo_r:.3f}, {hi_r:.3f}]  (lebar {100*(hi_r-lo_r):.0f} pp)")
print(f"Q1  tertangkap: {hasil['tertangkap_q1']}")
print(f"Q1  meleset   : {hasil['meleset_q1']}")
print()
print(f"Q2  TP {q2['tp']}  FP {q2['fp']}  FN {q2['fn']}   <== TIDAK SAHIH, acuan cuma {hasil['q2_n_acuan']} positif")
print('```')

```
ANGKA UNTUK ISSUE #2
model judge   : Qwen/Qwen2.5-7B-Instruct (backend hf, greedy, 2xT4)
sampel        : 260 baris, seed=7
ditandai judge: Q1 16, Q2 47

Q1  TP 7  FP 9  FN 3
Q1  presisi 0.438  n=16  CI95 [0.231, 0.668]  (BATAS BAWAH)
Q1  recall  0.700  n=10  CI95 [0.397, 0.892]  (lebar 50 pp)
Q1  tertangkap: [53, 108, 111, 144, 203, 220, 222]
Q1  meleset   : [25, 140, 215]

Q2  TP 0  FP 47  FN 1   <== TIDAK SAHIH, acuan cuma 1 positif
```


## 7. Kemas hasil untuk diunduh

Menghasilkan `hasil_kalibrasi_issue2.zip` di `/kaggle/working`. Klik panel **Output** di kanan
lalu unduh zip itu.

Isi zip diekstrak di laptop ke `FP_NLP/reports/kalibrasi_judge/`, lalu notebook
`notebooks/revisi/08_kalibrasi_judge.ipynb` dijalankan ulang — sel yang tadi berkata "BELUM ADA
VONIS JUDGE" akan terisi tabel + grafik.

In [10]:
import zipfile

berkas = sorted(OUT.glob('*'))
zip_path = Path('/kaggle/working/hasil_kalibrasi_issue2.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in berkas:
        z.write(f, arcname=f.name)

print(f'{zip_path}  ({zip_path.stat().st_size/1024:.0f} KB)')
for f in berkas:
    print(f'  {f.name:32} {f.stat().st_size/1024:8.1f} KB')

print()
print('LANGKAH BERIKUTNYA DI LAPTOP:')
print('  1. unduh hasil_kalibrasi_issue2.zip dari panel Output')
print('  2. unzip ke FP_NLP/reports/kalibrasi_judge/')
print('  3. lapor ke Claude — sisanya (notebook 08, komentar issue, commit) dikerjakan di sana')

/kaggle/working/hasil_kalibrasi_issue2.zip  (3 KB)
  kalibrasi_judge_report.json           1.1 KB
  kalibrasi_judge_report.txt            1.5 KB
  kalibrasi_q1.progress                 2.4 KB
  kalibrasi_q2.progress                 2.4 KB

LANGKAH BERIKUTNYA DI LAPTOP:
  1. unduh hasil_kalibrasi_issue2.zip dari panel Output
  2. unzip ke FP_NLP/reports/kalibrasi_judge/
  3. lapor ke Claude — sisanya (notebook 08, komentar issue, commit) dikerjakan di sana


In [11]:
# Cetak laporan lengkap sekali lagi supaya tersimpan di output notebook Kaggle
print((OUT / 'kalibrasi_judge_report.txt').read_text(encoding='utf-8'))

KALIBRASI JUDGE vs ANOTASI MANUAL
model judge     : Qwen/Qwen2.5-7B-Instruct
sampel          : 260 baris (seed=7)
dinilai Q1/Q2   : 260 / 260
ditandai judge  : Q1 16, Q2 47

Tahap A (regex, sebelum judge): 9 baris terbuang, 0 di antaranya juga positif manual Q1

Q1 keterjawaban soal — 10 positif manual
  TP   7   FP   9   FN   3
  presisi 0.438  (n= 16)  CI95 Wilson [0.231, 0.668]   <- BATAS BAWAH (lihat catatan label lemah)
  recall  0.700  (n= 10)  CI95 Wilson [0.397, 0.892]   <- lebar 50 poin persen
  tertangkap: [53, 108, 111, 144, 203, 220, 222]
  meleset   : [25, 140, 215]

Q2 kelayakan bentuk gold — 1 positif manual   <== TIDAK SAHIH, n positif terlalu kecil
  TP   0   FP  47   FN   1
  presisi 0.000  (n= 47)  CI95 Wilson [0.000, 0.076]   <- BATAS BAWAH (lihat catatan label lemah)
  recall  0.000  (n=  1)  CI95 Wilson [0.000, 0.793]   <- lebar 79 poin persen
  PERINGATAN: acuan Q2 hanya 1 baris positif. Angka presisi/recall di atas TIDAK layak dikutip sebagai performa Q2 — n-nya

## 8. Uji chat template (opsional, ~15 menit tambahan)

Menjawab: apakah Q2 gagal karena judge dijalankan tanpa chat template, atau memang di luar
kemampuannya?

Model dimuat sekali lagi dengan `pakai_chat_template=False` — meniru jalur vLLM yang menyodorkan
prompt mentah. Hasilnya disilangkan dengan bentuk kunci jawaban dan dibandingkan run utama.

**Yang sudah diketahui sebelum uji ini** (dari run pertama, dihitung ulang di sel 8b):
Q2 menandai angka polos 91% tapi Q1 pada run yang sama **rata** (10% / 6% / 7%). Kalau ketiadaan
template merusak secara global, Q1 mestinya ikut melenceng — nyatanya tidak. Jadi hipotesis
template sudah lemah sejak awal; uji ini menutup kemungkinan terakhirnya, bukan mengonfirmasi
tebakan.

Lewati sel ini kalau waktu sesi Kaggle mepet — hasil utama di sel 5-7 tidak bergantung padanya.

In [12]:
# 8a. Silang run UTAMA (dengan chat template) — dasar pembanding
from src.data_validation.judge_quality import _load_progress

q1_utama = cal.vonis_ke_posisi(sampel, _load_progress(OUT / 'kalibrasi_q1.progress'), 'Q1')
q2_utama = cal.vonis_ke_posisi(sampel, _load_progress(OUT / 'kalibrasi_q2.progress'), 'Q2')

silang_q2_utama = cal.silang_bentuk_jawaban(sampel, q2_utama)
print(cal.cetak_silang(silang_q2_utama, 'Q2 DENGAN chat template'))
print()
print(cal.cetak_silang(cal.silang_bentuk_jawaban(sampel, q1_utama),
                       'Q1 DENGAN chat template (pembanding: harus rata)'))

Q2 DENGAN chat template
kelas jawaban     TIDAK    YA     n  % ditandai
angka_polos          14    63    77         18%
teks_lain            16    48    64         25%
latex                17   102   119         14%
TOTAL                47   213   260         18%

Q1 DENGAN chat template (pembanding: harus rata)
kelas jawaban     TIDAK    YA     n  % ditandai
angka_polos           6    71    77          8%
teks_lain             4    60    64          6%
latex                 6   113   119          5%
TOTAL                16   244   260          6%


In [13]:
# 8b. Ulangi TANPA chat template. Progress ditulis ke folder terpisah supaya run utama aman.
OUT_AB = Path('/kaggle/working/hasil_kalibrasi_tanpa_template')
OUT_AB.mkdir(parents=True, exist_ok=True)

t0 = time.time()
hasil_ab = cal.run(
    Path('data/Final/easy_clean_v2.jsonl'), OUT_AB,
    judge_backend='hf', judge_model='Qwen/Qwen2.5-7B-Instruct',
    device_map='auto', batch_size=16, regenerasi=False,
    pakai_chat_template=False,          # <-- meniru jalur vLLM
)
print(f'\nselesai dalam {(time.time()-t0)/60:.1f} menit')

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


[Q1] 260 baris, 0 sudah dinilai, 260 sisa


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1012.81 MiB is free. Including non-PyTorch memory, this process has 13.57 GiB memory in use. Of the allocated memory 13.40 GiB is allocated by PyTorch, and 48.26 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# 8c. Putusan
q2_ab = cal.vonis_ke_posisi(sampel, _load_progress(OUT_AB / 'kalibrasi_q2.progress'), 'Q2')
silang_q2_ab = cal.silang_bentuk_jawaban(sampel, q2_ab)
print(cal.cetak_silang(silang_q2_ab, 'Q2 TANPA chat template'))

pa = silang_q2_utama['angka_polos']['persen_ditandai']
pb = silang_q2_ab['angka_polos']['persen_ditandai']
print(f'\nangka polos ditandai TIDAK: dengan template {pa:.0f}%  ->  tanpa template {pb:.0f}%')
print(f'total ditandai: {hasil["n_ditandai_judge_q2"]} -> {hasil_ab["n_ditandai_judge_q2"]} dari 260')

if abs(pa - pb) < 10:
    print('\nPUTUSAN: chat template BUKAN penyebabnya. Selisih < 10 pp.')
    print('Q2 memang di luar kemampuan judge 7B untuk tugas ini -> matikan Q2 di #3.')
else:
    print('\nPUTUSAN: chat template BERPENGARUH. Selisih >= 10 pp.')
    print('PERINGATAN: Q1 juga wajib dikalibrasi ulang — recall 0,900 terukur pada kondisi lama.')

## Batas sahih — ikut dikutip, jangan dibuang

1. **Q1 cuma 10 positif manual.** Interval Wilson 95% recall lebarnya 28-53 poin persen. Kutip
   intervalnya, bukan titik estimasinya.
2. **Q2 cuma 1 positif pasti (+2 ragu).** Tidak cukup untuk angka apa pun; laporan menandainya
   `TIDAK SAHIH`. Jangan masukkan ke paper sebagai klaim performa Q2.
3. **Baris tak-ditandai = label lemah** ("tampak wajar saat disapu", bukan terverifikasi benar).
   Maka FP yang terhitung adalah batas atas, dan presisi yang dilaporkan adalah batas bawah.
4. **Recall judge bukan recall filter.** Tahap A (regex) sudah membuang sebagian baris rusak
   sebelum judge melihatnya. Performa filter utuh = tahap A + judge.